# WW Prediction – Hybrid v3 LOEO
Pipeline modulare identica al monolite `WW_hybrid_v3_loeo.py`.

**Struttura:**
```
STEP 1  → compute_sensor_residuals
STEP 2  → create_base_features
STEP 3  → aggregate_by_cycle
STEP 4a → estimate_ww_period_per_engine  [ORIGINALE]
STEP 4b → add_hpc_ww_recovery_feature    [MathWorks]
STEP 4c → add_residual_shock_features    [ORIGINALE]
STEP 4d → add_periodic_and_residual_features
STEP 5  → select_features  (dentro LOEO, su train)
STEP 6  → run_loeo
STEP 7  → plot_results + save_results
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
# Assicurati che src/ sia nel path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.utils import load_config

# Feature engineering
from src.phm_rul.features.residuals    import compute_sensor_residuals
from src.phm_rul.features.base         import create_base_features
from src.phm_rul.features.aggregate    import aggregate_by_cycle
from src.phm_rul.features.periodic_fft import estimate_ww_period_per_engine
from src.phm_rul.features.recovery     import add_hpc_ww_recovery_feature
from src.phm_rul.features.shock        import add_residual_shock_features
from src.phm_rul.features.rolling      import add_periodic_and_residual_features

# Pipeline LOEO + reporting
from src.phm_rul.pipeline   import run_loeo
from src.phm_rul.reporting  import plot_results, save_results

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
cfg = load_config('../configs/config.yaml')

WINDOW_SIZE = 32
TOP_K       = 70
OUT_DIR     = f'../artifacts/hybrid_v3_loeo_{WINDOW_SIZE}'


## STEP 1–4: Feature Pipeline (snapshot-level)

In [ ]:
df = pd.read_csv(cfg['data']['train_clean_csv'])
print(f'Loaded: {df.shape}, ESNs: {sorted(df["ESN"].unique())}')

total_nan = df.isna().sum().sum()
if total_nan:
    print(f'  ⚠️  RAW: {total_nan} NaN found:')
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

In [ ]:
print('\n' + '='*70)
print('STEP 1: SENSOR RESIDUALS')
print('='*70)
df, res_cols = compute_sensor_residuals(df)

In [ ]:
print('\n' + '='*70)
print('STEP 2: BASE FEATURES')
print('='*70)
df = create_base_features(df)

In [ ]:
print('\n' + '='*70)
print('STEP 3: AGGREGATING BY CYCLE')
print('='*70)
df_agg = aggregate_by_cycle(df, res_cols)

In [ ]:
# STEP 4: tutte le feature originali
df_agg = estimate_ww_period_per_engine(df_agg)          # 4a FFT [ORIGINALE]
df_agg = add_hpc_ww_recovery_feature(df_agg)            # 4b MathWorks
df_agg = add_residual_shock_features(df_agg)            # 4c Shock [ORIGINALE]
df_agg = add_periodic_and_residual_features(df_agg, res_cols)  # 4d Rolling

print(f'\ndf_agg shape: {df_agg.shape}')

## STEP 5–6: LOEO
> `select_features` viene chiamata **dentro** `run_loeo` ad ogni fold, solo su `df_train`. Questo replica esattamente il comportamento del monolite e previene il leakage.

In [ ]:
fold_results = run_loeo(
    df_agg,
    res_cols    = res_cols,
    window_size = WINDOW_SIZE,
    top_k       = TOP_K,
)

## STEP 7: Visualizzazione e salvataggio

In [ ]:
plot_results(fold_results, OUT_DIR)
df_res = save_results(fold_results, OUT_DIR)
print(f'\n📁 Results saved → {OUT_DIR}/')

In [ ]:
# Riepilogo rapido
print('\n--- SUMMARY ---')
print(df_res[['left_out_esn','mae','twe','r2','improvement']].to_string(index=False))